# 03 — RQ2 Ablation: CLEAN Re-runs of Variants C and E on GPT-4o-mini

**Purpose.** Re-run Variants C (Patterns) and E (Full Framework) on the same 234 samples used in `02_run_ablation_hinted.ipynb`, but with the CWE-category language removed from the prompts. This isolates the contribution of category-hint contamination from prompt-structural effects, and supports the Answer Leakage analysis in RQ2 of the paper.

**Difference from `02_run_ablation_hinted.ipynb`.** The HINTED Variant C contains the phrase "SQL Injection or OS Command Injection", and the HINTED Variant E names "CWE-89 and CWE-78 patterns" explicitly. Both are removed in the CLEAN forms loaded by this notebook (`prompts/variant_C_patterns_clean.txt`, `prompts/variant_E_full_clean.txt`). All other settings (model, temperature, system message, samples) are held identical to enable a paired comparison.

**Inputs.**
- `<BASE_DIR>/final_dataset/` — produced by `01_dataset_curation.ipynb`.
- `prompts/variant_C_patterns_clean.txt`, `prompts/variant_E_full_clean.txt` — committed to the repository.
- `prompts/system_message.txt` — committed to the repository.
- An OpenAI API key with access to `gpt-4o-mini`.

**Outputs.**
- `<BASE_DIR>/results/rq2_ablation_results_clean.csv` — one row per sample, with two columns (`Variant_C_Patterns_CLEAN`, `Variant_E_Full_CLEAN`) containing the model's prediction.

**Cost & runtime.** 234 samples × 2 variants = 468 API calls on `gpt-4o-mini`. Approximate cost: USD 0.04–0.08. Approximate runtime: 12–18 minutes. Resume-on-interrupt is supported.


## 1. Setup

Same setup as `02_run_ablation_hinted.ipynb`: edit `BASE_DIR_OVERRIDE` if needed; `OPENAI_API_KEY` is read from the environment, falling back to interactive prompt.

In [ ]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ---- USER-EDITABLE ----
BASE_DIR_OVERRIDE = None  # e.g., Path('/content/drive/MyDrive/llm-vuln-detection-ablation')
MODEL_ID          = 'gpt-4o-mini'
TEMPERATURE       = 0.1
MAX_RETRIES       = 3
RETRY_BACKOFF_SEC = 2
# -----------------------

def find_repo_root() -> Path:
    """Locate the repository root by walking upward from the current working directory."""
    sentinels = ('README.md', 'requirements.txt', '.git')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if any((candidate / s).exists() for s in sentinels):
            return candidate
    return cwd

REPO_ROOT         = find_repo_root()
BASE_DIR          = Path(BASE_DIR_OVERRIDE).resolve() if BASE_DIR_OVERRIDE else (REPO_ROOT / 'workspace')
DATASET_DIR       = BASE_DIR / 'final_dataset'
RESULTS_DIR       = BASE_DIR / 'results'
PROMPTS_DIR       = REPO_ROOT / 'prompts'
OUTPUT_CSV        = RESULTS_DIR / 'rq2_ablation_results_clean.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

print(f'REPO_ROOT     : {REPO_ROOT}')
print(f'DATASET_DIR   : {DATASET_DIR}  (exists: {DATASET_DIR.exists()})')
print(f'PROMPTS_DIR   : {PROMPTS_DIR}  (exists: {PROMPTS_DIR.exists()})')
print(f'OUTPUT_CSV    : {OUTPUT_CSV}')
print(f'MODEL_ID      : {MODEL_ID}')


## 2. Load CLEAN prompt variants

Read the two CLEAN prompt variants and the system message from `prompts/`. Compare these against the HINTED forms loaded by `02_run_ablation_hinted.ipynb` to confirm that the only difference is the removal of CWE-category language.

In [ ]:
def load_prompt(filename: str) -> str:
    """Read a prompt file and strip a single trailing newline."""
    with open(PROMPTS_DIR / filename, encoding='utf-8') as f:
        return f.read().rstrip('\n')

PROMPTS = {
    'Variant_C_Patterns_CLEAN':  load_prompt('variant_C_patterns_clean.txt'),
    'Variant_E_Full_CLEAN':      load_prompt('variant_E_full_clean.txt'),
}
SYSTEM_MESSAGE = load_prompt('system_message.txt')

for name, text in PROMPTS.items():
    print(f'{name:30s} ({len(text):3d} chars): {text[:80]}{"..." if len(text) > 80 else ""}')
print()
print(f'{"SYSTEM_MESSAGE":30s} ({len(SYSTEM_MESSAGE):3d} chars): {SYSTEM_MESSAGE[:80]}...')


## 3. Run the CLEAN re-runs

For each of the 234 samples, query the model twice (once per CLEAN variant) and record the prediction. The output CSV is written incrementally and supports resume-on-interrupt.

In [ ]:
def get_true_label(filepath: Path) -> tuple:
    """Infer (true_label, true_cwe) from the parent directory name."""
    folder = filepath.parent.name
    if 'Safe' in folder:
        return 'Safe', None
    if '89' in folder:
        return 'Vulnerable', 'CWE-89'
    if '78' in folder:
        return 'Vulnerable', 'CWE-78'
    raise ValueError(f'Cannot infer label from folder name: {folder}')

def predict(prompt_text: str, code_content: str) -> str:
    """Send one (prompt, code) pair to the model with retry. Return 'Vulnerable' / 'Safe' / 'Error'."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                response_format={'type': 'json_object'},
                temperature=TEMPERATURE,
                messages=[
                    {'role': 'system', 'content': SYSTEM_MESSAGE},
                    {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
                ],
            )
            result = json.loads(response.choices[0].message.content)
            return result.get('prediction', 'Error')
        except Exception:
            if attempt + 1 < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC)
            else:
                return 'Error'

all_files = sorted(DATASET_DIR.rglob('*.php'))
if not all_files:
    raise RuntimeError(f'No .php files found under {DATASET_DIR}. Run 01_dataset_curation.ipynb first.')
print(f'Total samples: {len(all_files)}')

if OUTPUT_CSV.exists():
    results_df = pd.read_csv(OUTPUT_CSV)
    processed = set(results_df['File_Name'].tolist())
    print(f'Resuming from existing CSV: {len(processed)} samples already processed.')
else:
    columns = ['File_Name', 'True_Label', 'True_CWE'] + list(PROMPTS.keys())
    results_df = pd.DataFrame(columns=columns)
    processed = set()

to_process = [p for p in all_files if p.name not in processed]
print(f'Samples remaining to process: {len(to_process)}')

for filepath in tqdm(to_process, desc=f'CLEAN re-runs on {MODEL_ID}'):
    true_label, true_cwe = get_true_label(filepath)
    code_content = filepath.read_text(encoding='utf-8', errors='ignore')

    row = {'File_Name': filepath.name, 'True_Label': true_label, 'True_CWE': true_cwe}
    for variant_name, prompt_text in PROMPTS.items():
        row[variant_name] = predict(prompt_text, code_content)

    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\nCLEAN re-runs complete. Results written to {OUTPUT_CSV}')


## 4. Inspect results

Quick sanity checks on the output CSV. Detailed metric computation (paired McNemar tests against the HINTED results from `02`) is deferred to `05_metrics_and_figures.ipynb`.

In [ ]:
df = pd.read_csv(OUTPUT_CSV)

print(f'Total rows           : {len(df)}')
print(f'Expected             : 234')
print()
print('True label distribution:')
print(df['True_Label'].value_counts().to_string())
print()
print('True CWE distribution:')
print(df['True_CWE'].value_counts(dropna=False).to_string())
print()
print('Per-variant prediction distribution:')
for variant in PROMPTS.keys():
    counts = df[variant].value_counts().to_dict()
    print(f'  {variant:30s} {counts}')
